# Person B — Probability Calibration Analysis

This notebook **evaluates** how well existing Keras and TabNet probabilities match observed outcomes on **German Credit** and **Credit Card Fraud**. It does **not** fit a post-hoc calibrator to test data, which would cause data leakage.

**Input:** the two ZIP files you already downloaded:
- `person_b_results.zip` (Keras)
- `tabnet_person_b_results.zip` (TabNet)

**Output:** `calibration_person_b_results.zip` with summary CSVs and plots. Upload extracted outputs to `results/person_b/` and this notebook to top-level `person_b/`.

Run **Runtime → Run all**. The analysis should usually finish in a few minutes; it does not retrain the networks.


## 1 — Upload both saved results ZIPs

In [ ]:
from pathlib import Path
from google.colab import files
import zipfile, os

BASE = Path("/content/calibration_person_b")
OUT = BASE / "results"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)

for filename, dest in [("person_b_results.zip", BASE / "keras"),
                       ("tabnet_person_b_results.zip", BASE / "tabnet")]:
    dest.mkdir(parents=True, exist_ok=True)
    candidate = Path("/content") / filename
    if not candidate.exists():
        print(f"Please upload {filename}")
        uploaded = files.upload()
        if filename not in uploaded:
            raise FileNotFoundError(f"Expected {filename}, got {list(uploaded)}")
        candidate = Path(filename)
    with zipfile.ZipFile(candidate) as archive:
        archive.extractall(dest)
    print(f"Extracted {filename} → {dest}")

print("Ready:", sorted(p.name for p in (BASE / "keras").iterdir()))


## 2 — Load and validate matched predictions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import brier_score_loss, log_loss, average_precision_score, roc_auc_score

def load_german(root):
    f = pd.read_csv(root / "keras_german_25fold_predictions.csv" if root.name == "keras"
                    else root / "tabnet_german_25fold_predictions.csv")
    required = ["repeat", "fold", "row_index", "actual", "predicted_probability"]
    if any(col not in f.columns for col in required):
        raise ValueError(f"Missing columns in {root.name} German CSV")
    if f.duplicated(["repeat","fold","row_index"]).any():
        raise ValueError("Duplicate German prediction keys")
    return f[required].rename(columns={"predicted_probability":root.name})

kg = load_german(BASE / "keras")
tg = load_german(BASE / "tabnet")
german = kg.merge(tg, on=["repeat","fold","row_index","actual"], how="outer", validate="one_to_one", indicator=True)
if not (german["_merge"] == "both").all():
    raise ValueError("German fold predictions do not align between models")
german = german.drop(columns="_merge")
if len(german) != 5000 or german.groupby(["repeat","fold"]).size().nunique() != 1:
    raise ValueError("Unexpected German fold structure")

def load_fraud(root, model):
    f = np.load(root / f"{model}_fraud_predictions.npz")
    return {k:f[k] for k in f.files}

kf = load_fraud(BASE / "keras", "keras")
tf = load_fraud(BASE / "tabnet", "tabnet")
if not np.array_equal(kf["test_idx"], tf["test_idx"]) or not np.array_equal(kf["y_test"], tf["y_test"]):
    raise ValueError("Fraud test rows/labels do not match; cannot compare directly")
if kf["probabilities"].shape != tf["probabilities"].shape:
    raise ValueError("Fraud seed arrays have different shapes")

for name, arr in [("German Keras", german["keras"]), ("German TabNet", german["tabnet"]),
                  ("Fraud Keras", kf["probabilities"]), ("Fraud TabNet", tf["probabilities"])]:
    a = np.asarray(arr)
    if not np.isfinite(a).all() or ((a < 0) | (a > 1)).any():
        raise ValueError(f"Invalid probabilities in {name}")

print("German:", len(german), "test predictions across", german.groupby(["repeat","fold"]).ngroups, "folds")
print("Fraud:", len(kf["y_test"]), "test rows ×", len(kf["probabilities"]), "seeds")
print("Fraud positives:", int(kf["y_test"].sum()))


## 3 — Compute calibration metrics

- **Brier score:** mean squared error of probability estimates; lower is better.
- **Log loss:** penalizes confident incorrect probabilities; lower is better.
- **ECE (10 bins):** weighted average gap between observed event frequency and predicted probability. This is descriptive and sensitive to binning.
- **PR-AUC / ROC-AUC:** discrimination, not calibration.

For German Credit, metrics are computed **within each test fold**, then summarized across 25 folds. For fraud, each seed is evaluated on the same held-out temporal test set. Do not treat repeated folds or seeds as independent test datasets.


In [ ]:
def ece(y, p, bins=10, strategy="uniform"):
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)
    if strategy == "quantile":
        edges = np.unique(np.quantile(p, np.linspace(0, 1, bins + 1)))
        if len(edges) < 2:
            return float(abs(y.mean() - p.mean()))
        ids = np.clip(np.searchsorted(edges[1:-1], p, side="right"), 0, len(edges)-2)
        count = len(edges)-1
    else:
        ids = np.minimum((p * bins).astype(int), bins-1)
        count = bins
    return sum((mask.sum()/len(y))*abs(y[mask].mean()-p[mask].mean())
               for b in range(count) if (mask := (ids == b)).any())

def evaluate(y, p):
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)
    clipped = np.clip(p, 1e-12, 1-1e-12)
    return dict(brier=brier_score_loss(y,p),
                log_loss=log_loss(y,clipped,labels=[0,1]),
                ece10_uniform=ece(y,p,10,"uniform"),
                ece10_quantile=ece(y,p,10,"quantile"),
                pr_auc=average_precision_score(y,p),
                roc_auc=roc_auc_score(y,p))

rows = []
for (repeat,fold), group in german.groupby(["repeat","fold"]):
    for model in ["keras","tabnet"]:
        rows.append(dict(dataset="German Credit", model=model,
                         repeat=int(repeat), fold=int(fold), seed=np.nan,
                         **evaluate(group["actual"],group[model])))
for seed in range(len(kf["probabilities"])):
    for model, a in [("keras",kf),("tabnet",tf)]:
        rows.append(dict(dataset="Fraud",model=model,repeat=np.nan,fold=np.nan,seed=seed,
                         **evaluate(a["y_test"],a["probabilities"][seed])))

metrics = pd.DataFrame(rows)
metrics.to_csv(OUT/"calibration_per_run_metrics.csv",index=False)
summary = metrics.groupby(["dataset","model"])[["brier","log_loss","ece10_uniform","ece10_quantile","pr_auc","roc_auc"]].agg(["mean","std"])
summary.to_csv(OUT/"calibration_summary.csv")
print(summary.round(5).to_string())


## 4 — Paired Keras vs TabNet comparisons

In [ ]:
metric_cols = ["brier","log_loss","ece10_uniform","ece10_quantile","pr_auc","roc_auc"]
diff_rows = []
for dataset, keycols in [("German Credit",["repeat","fold"]),("Fraud",["seed"])]:
    m = metrics[metrics.dataset==dataset]
    a = m[m.model=="keras"].set_index(keycols)
    b = m[m.model=="tabnet"].set_index(keycols)
    if not a.index.equals(b.index):
        a,b = a.align(b,join="inner",axis=0)
    for metric in metric_cols:
        diff_rows.append(dict(dataset=dataset,metric=metric,
                              mean_keras=float(a[metric].mean()),
                              mean_tabnet=float(b[metric].mean()),
                              mean_tabnet_minus_keras=float((b[metric]-a[metric]).mean())))
diffs = pd.DataFrame(diff_rows)
diffs.to_csv(OUT/"calibration_paired_comparison.csv",index=False)
print(diffs.round(5).to_string(index=False))
print("Positive TabNet−Keras differences mean TabNet has a higher score;")
print("for Brier, log loss and ECE, LOWER is better.")


## 5 — Reliability diagrams (do not fit calibrators on these test predictions)

German Credit: combine **held-out predictions from repeat 0 only** (each applicant appears once). Fraud: use **seed 0** for both models. Using only one repeat avoids counting the same German borrower five times in a single reliability plot.

For fraud, show a zoomed probability axis because positive cases are very rare. We also save bin-level counts for transparency.


In [ ]:
def calibration_bins(y, p, n_bins=10, strategy="quantile"):
    y = np.asarray(y).astype(int); p = np.asarray(p,dtype=float)
    if strategy=="quantile":
        edges=np.unique(np.quantile(p,np.linspace(0,1,n_bins+1)))
        if len(edges)<2: edges=np.array([0.,1.])
    else:
        edges=np.linspace(0,1,n_bins+1)
    ids=np.searchsorted(edges[1:-1],p,side="right")
    records=[]
    for b in range(len(edges)-1):
        mask=ids==b
        if mask.any():
            records.append(dict(bin=b,n=int(mask.sum()),positives=int(y[mask].sum()),
                                mean_prediction=float(p[mask].mean()),
                                observed_rate=float(y[mask].mean())))
    return pd.DataFrame(records)

all_bins=[]
for dataset in ["German Credit","Fraud"]:
    if dataset=="German Credit":
        sub=german[german.repeat==0]
        y=sub["actual"].to_numpy()
        pred={"keras":sub["keras"].to_numpy(),"tabnet":sub["tabnet"].to_numpy()}
    else:
        y=kf["y_test"]
        pred={"keras":kf["probabilities"][0],"tabnet":tf["probabilities"][0]}
    fig,ax=plt.subplots(figsize=(7,6))
    for model,p in pred.items():
        bins=calibration_bins(y,p,10,"quantile")
        bins["dataset"]=dataset; bins["model"]=model
        all_bins.append(bins)
        ax.plot(bins["mean_prediction"],bins["observed_rate"],"o-",label=model)
    limit = max(0.01, max(float(np.max(pred[k])) for k in pred))
    if dataset=="Fraud":
        # Keep full x-axis to avoid concealing high predicted probabilities.
        ax.set_xlim(0,1);ax.set_ylim(0,1)
    else:
        ax.set_xlim(0,1);ax.set_ylim(0,1)
    ax.plot([0,1],[0,1],"--",color="gray",label="Perfect calibration")
    ax.set(xlabel="Mean predicted probability",ylabel="Observed positive-class frequency",
           title=f"{dataset}: test-set reliability (10 quantile bins)")
    ax.grid(alpha=.25);ax.legend()
    fig.tight_layout()
    fig.savefig(FIG/("calibration_german_reliability.png" if dataset=="German Credit" else "calibration_fraud_reliability.png"),dpi=180)
    plt.show()
pd.concat(all_bins,ignore_index=True).to_csv(OUT/"calibration_bin_details.csv",index=False)


## 6 — Compare per-run Brier scores

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4))
for ax,dataset in zip(axes,["German Credit","Fraud"]):
    sub=metrics[metrics.dataset==dataset]
    ax.boxplot([sub[sub.model==model]["brier"].values for model in ["keras","tabnet"]],
               tick_labels=["Keras","TabNet"])
    ax.set(title=dataset,ylabel="Brier score (lower is better)")
    ax.grid(axis="y",alpha=.25)
fig.tight_layout()
fig.savefig(FIG/"calibration_brier_comparison.png",dpi=180)
plt.show()


## 7 — Export a short interpretation and download everything

**Important limitation:** This notebook evaluates *existing probabilities*. It does **not** implement Platt scaling, isotonic regression or temperature scaling. Those require a dedicated validation/calibration set and fresh predictions from each fitted model on that set. Never fit such a calibrator on the test labels used here.

When writing the report, avoid treating repeated CV folds or different seeds as independent samples. For fraud, class weighting can change probability calibration.


In [ ]:
lines = [
    "# Person B — Calibration evaluation",
    "",
    "Existing held-out test probabilities were analyzed without fitting a calibrator on test labels.",
    "German Credit: 25 repeated stratified folds per model. Fraud: 5 training seeds on the same temporal test set.",
    "Lower Brier, log loss, and ECE indicate better calibration-related scores; ECE depends on binning.",
    "PR-AUC and ROC-AUC measure ranking/discrimination, not calibration.",
    "",
    "## Mean results",
]
for (dataset,model), group in metrics.groupby(["dataset","model"]):
    lines.append(f"- {dataset}, {model}: Brier={group.brier.mean():.5f}, "
                 f"log loss={group.log_loss.mean():.5f}, "
                 f"ECE(quantile)={group.ece10_quantile.mean():.5f}, "
                 f"PR-AUC={group.pr_auc.mean():.5f}.")
lines += [
    "",
    "## Limitations",
    "- No post-hoc calibrator was fitted; do not describe this as calibrated model performance.",
    "- German repeated folds overlap across repeats; fraud seeds share the same test observations.",
    "- Rare fraud events make reliability bins noisy. Inspect bin counts in calibration_bin_details.csv.",
    "- Model training and class weighting may differ, so interpret framework differences cautiously."
]
(OUT/"calibration_interpretation.md").write_text("\n".join(lines),encoding="utf-8")
import shutil
from google.colab import files
archive=shutil.make_archive("/content/calibration_person_b_results","zip",root_dir=str(OUT))
print("Saved:",archive)
files.download(archive)
